In [ ]:
#importing libraries
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

In [ ]:
data = pd.read_json("preprocessed_data.jsonl",lines=True)
data.sample(10)
#we are going to work on arabic (src)
data = data[['src',"label"]].rename(columns={'src':'text'})
data


,text,label
0,وي انا تعجبني برشا,pos
1,تبارك اله عليه,pos
2,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة,pos
3,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...,pos
4,محلاها,pos
...,...,...
17049,شبعة خمج.اينما يقيم الجهل,neg
17050,الانسان هذا مافهمتش شنوه دينو وشنوه انتماءاتو ...,neg
17051,عار عليكم اليوم تجيبوا هالمناظر لتلفزة,neg
17052,ما اخيبها سهرية خاطر انتي فيها يا مقداد,neg


In [ ]:
data['label'] = data['label'].apply(lambda x: 1 if x == 'pos' else 0)
#remove dots :

data['text'] = data['text'].str.replace('.', '')
data.sample(10)

,text,label
3379,برافو برافو برافو برافو برافو برافو برافو براف...,1
14903,واله خير من نوفل الورتاني,1
2657,لطفي يعطيك الصحة شرفت تونس وحدك الحاصيلو,1
7911,قانة كفرة,0
8650,تافه تفوه عليك,0
12139,قلت حياء أما ثلث من الواقع لاسف,0
15736,نهار كامل و هوى يتسردك علينا هذا -_-,0
1427,فرامون سيتكوم روعة ضحك بالدموع,1
14801,معلم سامي,1
13926,برافو خالد شوكات,1


In [ ]:
#printing the number of training and testing positive and negative text
train_data = data.sample(frac=0.8, random_state=42)
test_data = data.drop(train_data.index)

pos_train_text = train_data[train_data['label'] == 1]['text'].tolist()
neg_train_text = train_data[train_data['label'] == 0]['text'].tolist()

print("Number of training positive text: ", len(pos_train_text))
print("Number of training negative text: ", len(neg_train_text))


Number of training positive text:  7108
Number of training negative text:  6535


In [ ]:
#tokinize by space
def tokenize(text):
    return text.split()

data['tokinized_text'] = data['text'].apply(tokenize)
data.sample(100)

,text,label,tokinized_text
13489,ماسط,0,[ماسط]
16300,جبالك الرواسي و لا المنظر هذا,0,"[جبالك, الرواسي, و, لا, المنظر, هذا]"
15572,ماسط,0,[ماسط]
7556,طيح سعدك حنان,0,"[طيح, سعدك, حنان]"
2086,وحدك في الوهرة يا أمين قارة,1,"[وحدك, في, الوهرة, يا, أمين, قارة]"
...,...,...,...
8426,تفوه على اولاد الكلبة,0,"[تفوه, على, اولاد, الكلبة]"
7086,انت معلم يسكتو وأنت موجود ميقدرو يتكلمو وله معلم,1,"[انت, معلم, يسكتو, وأنت, موجود, ميقدرو, يتكلمو..."
4056,يا خويا يعجبني,1,"[يا, خويا, يعجبني]"
5435,يقتل,1,[يقتل]


In [ ]:
#stemming
from nltk.stem.isri import ISRIStemmer
stemmer = ISRIStemmer()
def stem_words(words):
    return [stemmer.stem(word) for word in words]

data['stemmed_text'] = data['tokinized_text'].apply(stem_words)
data.sample(10)

#the arabic stemmer doesn't work very well on tunisian arabic so im going to train two models one for the tokinized text and another for the stemmed text

,text,label,tokinized_text,stemmed_text
15371,تفو عليكم تعلموا في بناتكم الرخس وقلة التربية ...,0,"[تفو, عليكم, تعلموا, في, بناتكم, الرخس, وقلة, ...","[تفو, علي, علم, في, بنت, رخس, وقل, ربي, رملم, ..."
2102,كافون يا معلم ديما محفلها و عاملنا جو برافو,1,"[كافون, يا, معلم, ديما, محفلها, و, عاملنا, جو,...","[كاف, يا, علم, ديم, حفل, و, عمل, جو, رفو]"
14480,ما نبالغشكنقول تونس كاملة تحبها كمثلة ناجحة وم...,1,"[ما, نبالغشكنقول, تونس, كاملة, تحبها, كمثلة, ن...","[ما, نبالغشكنقول, ونس, كمل, تحب, كمثل, نجح, وم..."
6495,بصراحة نور شيبا برافو,1,"[بصراحة, نور, شيبا, برافو]","[صرح, نور, شيب, رفو]"
7974,وجيهة انحشى على عصبها شوية وسع بالك يا وجيهة,0,"[وجيهة, انحشى, على, عصبها, شوية, وسع, بالك, يا...","[وجه, حشى, على, عصب, شوة, وسع, بلك, يا, وجه]"
6096,جعفور المعلم و راجل,1,"[جعفور, المعلم, و, راجل]","[جعفور, علم, و, رجل]"
2339,معلم و حافظوا عليه,1,"[معلم, و, حافظوا, عليه]","[علم, و, حفظ, عليه]"
10372,ساقطة إيناس,0,"[ساقطة, إيناس]","[سقط, ينس]"
16304,حسيت روحي بهمت,0,"[حسيت, روحي, بهمت]","[حست, روح, بهم]"
13996,محبوب وباهي وتستاهل كل خير,1,"[محبوب, وباهي, وتستاهل, كل, خير]","[حبب, وبه, تهل, كل, خير]"


In [ ]:
#printing the number of training and testing positive and negative text
train_data = data.sample(frac=0.8, random_state=42)
test_data = data.drop(train_data.index)

pos_train_text = train_data[train_data['label'] == 1]['text'].tolist()
neg_train_text = train_data[train_data['label'] == 0]['text'].tolist()

print("Number of training positive text: ", len(pos_train_text))
print("Number of training negative text: ", len(neg_train_text))


pos_train_corpus = " ".join(pos_train_text)
neg_train_corpus = " ".join(neg_train_text)


Number of training positive text:  7108
Number of training negative text:  6535


In [ ]:
#building the vocabulary from the tokinized
corpus = " ".join(data["text"] )
vocab_tokinized = set(corpus.split())
vocab_size_tokinized = len(vocab_tokinized)
print("Vocabulary size: ", vocab_size_tokinized)

#building the vocbulary from the stemmed

vocab_stemmed = set(
    w
    for _, row in data.iterrows()
    for w in row["stemmed_text"])




vocab_size_stemmed = len(vocab_stemmed)
print("Vocabulary size: ", vocab_size_stemmed)

Vocabulary size:  20354
Vocabulary size:  8218


In [ ]:
#let's start with tokenized model
#frequency dict
freq = dict()
for w in vocab_tokinized:
  positive_occ = pos_train_corpus.count(w)
  negative_occ = neg_train_corpus.count(w)
  freq[w] = (positive_occ,negative_occ)





In [ ]:
def word2freq(w:str):
  return freq.get(w,(0,0))

In [ ]:
for w in vocab_tokinized:
  pos,neg = word2freq(w)
  print(f"word : {w}  pos : {pos}  neg : {neg} ")

word : وضع  pos : 2  neg : 8 
word : مالثورة  pos : 0  neg : 1 
word : نعلة  pos : 0  neg : 7 
word : 10000  pos : 3  neg : 0 
word : تقن  pos : 3  neg : 5 
word : المتربين  pos : 0  neg : 1 
word : يعارض  pos : 1  neg : 0 
word : عالريمة  pos : 1  neg : 0 
word : ابهيم  pos : 0  neg : 6 
word : ملامهزلةه  pos : 0  neg : 1 
word : -)  pos : 57  neg : 2 
word : فوري  pos : 0  neg : 0 
word : نزيدو  pos : 0  neg : 2 
word : الحمراوي  pos : 2  neg : 0 
word : يسكت  pos : 2  neg : 2 
word : 10000000000000000000000  pos : 1  neg : 0 
word : نعتزل  pos : 1  neg : 0 
word : الحسد  pos : 3  neg : 1 
word : والتفاهة  pos : 0  neg : 3 
word : الخفاش  pos : 0  neg : 1 
word : نرحموك  pos : 0  neg : 1 
word : التوت  pos : 0  neg : 1 
word : وفن  pos : 4  neg : 0 
word : تجريح  pos : 1  neg : 1 
word : همني  pos : 0  neg : 1 
word : معاش  pos : 1  neg : 5 
word : حيو  pos : 1  neg : 75 
word : لأنذال  pos : 0  neg : 0 
word : معمري  pos : 0  neg : 0 
word : تبندير  pos : 1  neg : 5 
word : الميثالي

In [ ]:
train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

In [ ]:
#building the trainiig dataset
matrix_df_train = np.zeros((len(train_data),3))
for i ,row in train_data.iterrows():

  matrix_df_train[i,2] = row['label']
  for w in row['tokinized_text']:
    pos,neg = word2freq(w)
    matrix_df_train[i,0] += pos
    matrix_df_train[i,1] += neg



In [ ]:
df_train = pd.DataFrame(matrix_df_train,columns = ['positive_count' , 'negative_count',"label"])

In [ ]:
df_train

,positive_count,negative_count,label
0,7184.0,9391.0,0.0
1,12771.0,13625.0,1.0
2,1142.0,2955.0,0.0
3,164.0,1.0,1.0
4,2237.0,1457.0,1.0
...,...,...,...
13638,1836.0,1265.0,1.0
13639,2.0,290.0,0.0
13640,30685.0,33158.0,1.0
13641,829.0,23.0,1.0


In [ ]:
#building the trainiig dataset
matrix_df_test = np.zeros((len(test_data),3))
for i ,row in test_data.iterrows():

  matrix_df_test[i,2] = row['label']
  for w in row['tokinized_text']:
    pos,neg = word2freq(w)
    matrix_df_test[i,0] += pos
    matrix_df_test[i,1] += neg



In [ ]:
df_test = pd.DataFrame(matrix_df_test,columns = ['positive_count' , 'negative_count',"label"])

In [ ]:
df_train

,positive_count,negative_count,label
0,7184.0,9391.0,0.0
1,12771.0,13625.0,1.0
2,1142.0,2955.0,0.0
3,164.0,1.0,1.0
4,2237.0,1457.0,1.0
...,...,...,...
13638,1836.0,1265.0,1.0
13639,2.0,290.0,0.0
13640,30685.0,33158.0,1.0
13641,829.0,23.0,1.0


In [ ]:
X_train= df_train.drop('label',axis=1).to_numpy()
Y_train = df_train['label'].to_numpy()

X_test = df_test.drop('label',axis=1).to_numpy()
Y_test = df_test['label'].to_numpy()



In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression()

y_pred = log_reg.fit(X_train, Y_train).predict(X_test)
accuracy = log_reg.score(X_test, Y_test)

In [ ]:
print(f"accuracy : {accuracy:.4f}")

accuracy : 0.7282


In [ ]:
def inference(text):
  #i need to make the preprocessing a function
  tokinized_text = tokenize(text)
  positive_counts = 0
  negative_counts =0

  for w in tokinized_text:
    pos,neg = word2freq(w)
    positive_counts += pos
    negative_counts += neg

  features = np.array([[positive_counts,negative_counts]])
  return f"log_reg.predict(features)[0].item()"



In [ ]:
GREEN = "\033[92m"
RED = "\033[91m"
CYAN = "\033[96m"
YELLOW = "\033[93m"
RESET = "\033[0m"

for i, row in test_data.iterrows():

    text = row["text"]
    actual_label = row["label"]
    pred_label = inference(text)

    color = GREEN if actual_label == pred_label else RED
    status = "✓ CORRECT" if actual_label == pred_label else "✗ WRONG"

    print(f"{CYAN}Text:{RESET} {text}")
    print(f"{YELLOW}Actual label:{RESET} {actual_label}")
    print(f"{CYAN}Predicted label:{RESET} {pred_label}")
    print(f"{color}{status}{RESET}")
    print("-" * 80)

Streaming output truncated to the last 5000 lines.
Text: فلو
Actual label: 0
Predicted label: 1.0
✗ WRONG
--------------------------------------------------------------------------------
Text: الشكسرة والبحر
Actual label: 0
Predicted label: 1.0
✗ WRONG
--------------------------------------------------------------------------------
Text: مصطها اليلة ماقر شكيب
Actual label: 0
Predicted label: 1.0
✗ WRONG
--------------------------------------------------------------------------------
Text: عبيطة
Actual label: 0
Predicted label: 1.0
✗ WRONG
--------------------------------------------------------------------------------
Text: عيب عليك الناس تفرج في ديارهم عائلة معاهم بنات عيب عليك أنتي راجل كبير و ناس الكل كانت تقدرك أي كانت خاطر بعد الحلقة باش توالي تفريكة 2016
Actual label: 0
Predicted label: 0.0
✓ CORRECT
--------------------------------------------------------------------------------
Text: مابسلو عبد توا هذا يضحك بجاه ربي 
Actual label: 0
Predicted label: 1.0
✗ WRONG
----------------